In [14]:
import json
import numpy as np
import shutil
import pandas as pd

In [2]:
df = pd.read_csv('mhc_data/TCR3d_data.csv')
df['PDB ID'] = df['PDB ID'].astype(str).str.lower().str.strip()
df['Release date'] = pd.to_datetime(df['Release date'])

In [3]:
mask_train = df['Release date'] < '2021-09-30'
train_df = df[mask_train]

In [4]:
len(train_df)

1082

In [5]:
mask_val = (df['Release date'] >= '2021-09-30') & (df['Release date'] < '2023-01-13')
val_df = df[mask_val]

In [6]:
len(val_df)

130

In [10]:
train_ids = train_df['PDB ID'].dropna().tolist()
val_ids = val_df['PDB ID'].dropna().tolist()

In [40]:
# Apenas para guardar essa informação, não é necessário rodar esse código novamente
# No resto do código vamos usar o train_ids e val_ids

with open('mhc_data/train_templated.txt', 'w') as f:
    f.write('\n'.join(train_ids))

with open('mhc_data/val_templated.txt', 'w') as f:
    f.write('\n'.join(val_ids))

In [7]:
with open('rcsb_processed_targets/manifest.json') as f:
    data = json.load(f)

In [11]:
with open('mhc_data/train_templated.json', 'w') as f:
    data_to_dump = [sample for sample in data if sample['id'] in train_ids]
    json.dump(data_to_dump, f)

with open('mhc_data/val_templated.json', 'w') as f:
    data_to_dump = [sample for sample in data if sample['id'] in val_ids]
    json.dump(data_to_dump, f)

In [22]:
len(data)

216870

In [12]:
with open('mhc_data/train_templated.json') as f:
    train = json.load(f)

with open('mhc_data/val_templated.json') as f:
    val = json.load(f)

In [73]:
train_val = pd.concat([train_df, val_df], ignore_index=True)
train_val = train_val.to_dict(orient='records')

In [78]:
len(train_val)
train_val[0]

{'PDB ID': '1a1m',
 'MHC allele': 'HLA-B*53',
 'Species': 'Human',
 'Peptide*': 'TPYDINQML',
 'Bound to TCR': 1.0,
 'Release date': Timestamp('1998-04-07 00:00:00'),
 'Pubmed': 8624812.0,
 'Resolution': 2.3}

In [ ]:
# train_val = [sample for sample in train_val if sample['PDB ID']=='1a1m']

In [15]:
manifest_dict = {sample['id']: sample for sample in data}

In [ ]:
manifest_dict = {sample['id']: sample for sample in data}

In [ ]:
target_dir = 'mhc_samples'

import os
os.makedirs(target_dir, exist_ok=True)
os.makedirs(f"{target_dir}/msa/", exist_ok=True)
os.makedirs(f"{target_dir}/structures/", exist_ok=True)

new_manifest = []

bad_ids = []

for sample in train_val:
    if sample['PDB ID'] not in manifest_dict:
        bad_ids.append(sample['PDB ID'])
        continue
    sample_manifest = manifest_dict[sample['PDB ID']]
    peptide_chain_name = sample['peptide_chain'] + '1'
    protein_chain_name = sample['protein_chains'][0] + '1'
    matched_chains = 0
    valid_chain_ids = []
    for chain in sample_manifest['chains']:
        if chain['msa_id'] != -1:
            shutil.copy(f"rcsb_processed_msa/{chain['msa_id']}.npz", f"{target_dir}/msa/{chain['msa_id']}.npz")
        if chain['chain_name'] in [peptide_chain_name, protein_chain_name]:
            matched_chains += 1
            valid_chain_ids.append(chain['chain_id'])
        else:
            chain['valid'] = False
    if matched_chains != 2:
        bad_ids.append(sample['PDB ID'])
        # print('Didnt find both chains', sample['PDB ID'])
    else:
        n_correct_interfaces = 0
        for interface in sample_manifest['interfaces']:
            if (interface['chain_1'] in valid_chain_ids) and (interface['chain_2'] in valid_chain_ids):
                n_correct_interfaces += 1
            else:
                interface['valid'] = False
        if n_correct_interfaces != 1:
            print('Number of correct interfaces is wrong:', sample['id'], n_correct_interfaces)
        else:
            new_manifest.append(sample_manifest)
            
            # update mask in npz just in case
            npz = dict(np.load(f"rcsb_processed_targets/structures/{sample_manifest['id']}.npz"))
            for chain_id in range(len(npz['mask'])):
                if chain_id in valid_chain_ids:
                    npz['mask'][chain_id] = True
                else:
                    npz['mask'][chain_id] = False
            np.savez(f"{target_dir}/structures/{sample_manifest['id']}", **npz)
    # break

KeyError: 'protein_chains'

In [17]:
with open(f"{target_dir}/manifest.json", "w") as outfile:
    outfile.write(json.dumps(new_manifest))

In [18]:
len(train_val), len(bad_ids)

(1, 0)

In [98]:
val_list = [sample['pdb_id'].upper() for sample in val]

with open("{target_dir}/validation_ids.txt", "w") as outfile:
    outfile.write('\n'.join(val_list))

In [118]:
# !zip -r mhc.zip mhc_targets/

In [115]:
# scp mhc.zip eglukhov@nabu5.ams.stonybrook.edu:/home/eglukhov/projects/boltz/train_data/

In [ ]:
< 2021-09-30 - train
< 2023-01-13 - val
> - test